# Software Bug Prediction

### Dataset:
The dataset is collected at the University of Geneva in Switzerland related to the bug prediction dataset. It contains data about the following software systems:
    - Eclipse JDT Core
    - Eclipse PDE UI
    - Equinox Framework
    - Lucene
    - Mylyn
    
### Objective:
Software success depends heavily on the ability of maintaining the software and ensuring that the software is released with minimal number of issues. Knowing in advance an estimation of possibile bugs in the software will mitagate some of the risks after the software is released. Our main objective is to be able to predict in advance the number of bugs in the software using the software data itself such as the number of lines of code, number of methods, number of attributes, and other important software properties. This is an overview of our objectives in details:
- Data Analysis
- Data preprocessing
- Data visualization
- Advanced EDA and visulaization using ML/D-reduction algorithms
- hyper-parameter tuning and solving the imbalance problem
    - Classifying data where the classes are: no bugs, 1 bug, or > 2 bugs
    - Classifying data where the classes are: no bugs, > 0 bugs

### Outline:
- import
- data cleaning
- EDA and visualization
    - descriptive statistics
    - correlation matrix
    - feature importance (Lasso)
    - feature importance (RFC)
    - kernal density estimation
    - 3D scatter cross plots
    - UMAP dimensionality reduction
    - PCA dimensionality reduction
    - clustering analysis
   
   
- Base Line Classifier
    - Accuacy
    - ROC
    - F-1 Score
    - Confusion Matrix
    - Area Under the Curve of ROC Viz
    
    
- first stage modeling
    - hyper-parameter tuning model optimization
    - dimensionality reduction
    
    
- evaluation
    - Accuacy
    - ROC
    - F-1 Score
    - Confusion Matrix
    - Area Under the Curve of ROC Viz
    
    
- second stage modeling (data-driven model optimization to tackle imbalance)
    - Under Sampling
    - Over Sampling
    - Feature Selection


- evaluation
    - Accuacy
    - ROC
    - F-1 Score
    
    
- models quality metrics table
    - Multi
        - Accuacy (All and Best)
        - ROC (All and Best)
        - F-1 Score (All and Best)
    - Binary
        - Accuacy (All and Best)
        - ROC (All and Best)
        - F-1 Score (All and Best)

In [1]:
import pandas as pd
import numpy as np
import random as rnd

import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (8, 5)

from sklearn.ensemble import BaggingClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import Perceptron
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

from sklearn.ensemble import AdaBoostClassifier


from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, auc, confusion_matrix, f1_score

from sklearn.preprocessing import LabelBinarizer


from mpl_toolkits.mplot3d import Axes3D
from sklearn.cluster import KMeans
from sklearn.dummy import DummyClassifier
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
import umap


ModuleNotFoundError: No module named 'umap'

# Import

In [ ]:
eclipse_jdt = pd.read_csv("./data/Eclipse_ JDT_Core_single-version-ck-oo_bugs_only.csv")
eclipse_pdt = pd.read_csv("./data/Eclipse_PDE_UI_single-version-ck-oo_bug_only.csv")
equinox = pd.read_csv("./data/Equinox_Framework_single-version-ck-oo_bug_only.csv")
lucene = pd.read_csv("./data/Lucene_single-version-ck-oo_bug_only.csv")
mylyn = pd.read_csv("./data/Mylyn_single-version-ck-oo_bug_only.csv")

# Preprocessing & Analysis

In [ ]:
print("NaNs in eclipse_jdt", np.sum(np.sum(eclipse_jdt.isna(), axis=0)) )
print("NaNs in eclipse_pdt", np.sum(np.sum(eclipse_pdt.isna(), axis=0)) )
print("NaNs in equinox", np.sum(np.sum(equinox.isna(), axis=0)) )
print("NaNs in lucene", np.sum(np.sum(lucene.isna(), axis=0)) )
print("NaNs in mylyn", np.sum(np.sum(mylyn.isna(), axis=0)) )

# drop nans
eclipse_pdt.dropna(axis = 1, inplace=True)
equinox.dropna(axis = 1, inplace=True)
lucene.dropna(axis = 1, inplace=True)
mylyn.dropna(axis = 1, inplace=True)

### Note: 
All Nans are added columns due to added commas (seperators) in each row

In [ ]:
print("Data Shapes:", eclipse_jdt.shape, eclipse_pdt.shape, equinox.shape, lucene.shape, mylyn.shape)
df = pd.concat([eclipse_jdt, eclipse_pdt, equinox, lucene, mylyn], ignore_index=True)
df.columns = df.columns.str.replace(' ', '')
print("Full dataframe shape:",df.shape, '\n')
print("Predictors:")
for name in df.columns.values[1:18].tolist():
    print(name, end=', ')
print("\n\nPredictable:", df.columns.values[18])
df.head()

In [ ]:
df.describe()

### Note: 
- Values in columns are distributed with high standard deviation compared to others -> Scaling is needed if a distance-based model is used, but not now on the whole dataset to prevent data leakage.
- We do not need confusing variables such as the class name; We can use it as an index or discard it.


In [ ]:
# Shuffle data before removing classname to keep mapping
df = df.sample(frac=1.0)

# We do not need confusing variables such as the class name; We can use it as an index or leave it.
X = df.iloc[:, 1:-2]
y = df["bugs"]

print("X:", X.shape)
print("y:", y.shape)

In [ ]:
X.head()

In [ ]:
X = df.iloc[:, 1:-2]
y = df["bugs"]

print("X:", X.shape)
print("y:", y.shape)

### Make Classes [0, 1, 2] Bugs

for Multi-classifiers; we will start with Task 2 first

In [ ]:
# Warning: Don't Run Twice
y = y.replace(y.where(y > 2), 2)

In [ ]:
print("X:", X.shape)
print("y:", y.shape)

In [ ]:
((y.index == X.index).sum())

### Class Balance

In [ ]:
unique, counts = np.unique(y, return_counts=True)
print("Classes:", unique.tolist())
print("Counts:", counts.tolist())

plt.bar(unique, counts, color=['g', 'orange', 'r'], alpha=0.7)
plt.title("#Bugs VS Occurrences")
plt.xticks(range(len(unique)))
plt.ylabel("Occurrences")
plt.xlabel("# Bugs");

### Note: 
Based on these class balances, we have to be careful not to overfit or make the classifier favors "Class 0" unfairly

# More EDA on Training Set

### Note: 
It is better to Leave-One-Out and do the EDA & some transoformational preprocessing so that we do not use our intuition and analysis from the test set  and be fooled by a high accuracy on a set that we have prepared our model for.

###### Example: 
Feature Selection that is done on the whole dataset is a very pure data leakage. Your test set is not a test set if the model has already known what are the best features to use as predictors of the target value

###### Example: 
Dimensionality reduction (such as PCA) that is done one the whole dataset means that you have leaked distributional information from the test set to the training set (e.g. Variance). 

###### Example: 
Scaling the dataset as a whole is needed but when you use the whole dataset for fitting the scalar, then data leakage has occured, if a blueprint dataset factor (e.g. variance, mean, minimum, maximum) is used

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
X_cv, X_test, y_cv, y_test = train_test_split(X_test, y_test, test_size=0.5)
print("Train:", X_train.shape, y_train.shape,
      "Test:", X_test.shape, y_test.shape,
      "Cross Validation", X_cv.shape, y_cv.shape)

Let's make sure that we have a similar class balance distribution

In [ ]:
unique, counts = np.unique(y_train, return_counts=True)
print("Classes:", unique.tolist())
print("Counts:", counts.tolist())

plt.bar(unique, counts, color=['g', 'orange', 'r'], alpha=0.7)
plt.title("#Bugs VS Occurrences")
plt.xticks(range(len(unique)))
plt.ylabel("Occurrences")
plt.xlabel("# Bugs");

##### Scaling Features

In [ ]:
X_train_scaled = pd.DataFrame(StandardScaler().fit_transform(X_train.values), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(StandardScaler().fit_transform(X_test.values), columns=X_test.columns, index=X_test.index)

data_for_viz = X_train_scaled.copy()
data_for_viz_unscaled = X_train.copy()

data_for_viz['Bugs'] = y_train.copy().tolist()
data_for_viz_unscaled['Bugs'] = y_train.copy().tolist()

In [ ]:
data_for_viz.head()

#### Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10,8))

corr = data_for_viz.corr()
mask = np.triu(corr)
sns.heatmap(corr, vmin=-1, vmax=1, center= 0, cmap= 'Pastel2', linewidths=3, ax=ax, mask=mask);

###### We can see that highest correlated variables with Bugs are: 
1. cbo
    - is correlated with fanIn
    - is correlated with fanOut
2. fanout
    - is correlated with wmc
    - is correlated with rfc
    - is correlated with numberOfLinesOfCode
    - is correlated with numberOfMethods
3. numberOfLinesOfCode
    - is correlated with wmc
    - is correlated with rfc
    - is correlated with numberOfMethods
4. numberOfMethods
    - is correlated with wmc
    - is correlated with rfc
    - is correlated with numberOfPublicMethods
    
    - .
    - .
    - .
    

#### To sum up: There is a high correlation between: 
    - Bugs
    - cbo
    - fanout
    - fanIn
    - wmc
    - rfc
    - numberOfLinesOfCode
    - numberOfMethods
    - numberOfPublicMethods
    - numberOfPrivateMethods

### Feature importance with Lasso regression

In [ ]:
lasso = Lasso()
lasso.fit(X_train,y_train)
coef = pd.Series(lasso.coef_, index = X_train.columns)

print("Discarded Features:", np.sum(lasso.coef_==0), "out of", len(X_train.columns))

In [ ]:
coef = coef[coef != 0]

In [ ]:
coef.sort_values().plot(kind='barh', cmap="Pastel2");

### Feature importance with Random Forest Classifier

In [ ]:
rfc = RandomForestClassifier()
rfc.fit(X_train, y_train);
coef_rfc = pd.Series(rfc.feature_importances_, index = X_train.columns)

In [ ]:
preds = rfc.predict(X_train)
print("accuracy score:", accuracy_score(y_train, preds))

### Note: 
of course, the accuracy is going to be high on the same set that was trained on, we are just making sure that it has done a good job on predicting the training set and thus giving an accurate prediction of important features **based on the training set**

In [ ]:
coef_rfc.sort_values()[7:].plot(kind='barh', cmap="Pastel2");

#### Pick Features for DataViz
Let's now pick a couple of important features from our analysis with correlation matrix, lasso regression feature importance, and random forest classifier feature importance, we will go with:
    - rfc
    - cbo
    - fanOut
    - wmc
    - numberOfLinesOfCode
#### As these features are the most effective ones as predictors over our analysis
# Data Visualization   

In [ ]:
data_for_viz_unscaled.head()

#### Pair Plots (Scatter)

In [ ]:
for_pair_plot = data_for_viz_unscaled[['rfc', 'cbo', 'fanOut','wmc', 'numberOfLinesOfCode', 'Bugs']]
pairplot = sns.pairplot(for_pair_plot, hue="Bugs", vars=['rfc', 'cbo', 'fanOut', 'wmc', 'numberOfLinesOfCode']);
pairplot.fig.suptitle("Scatter w.r.t classes: 0, 1, > 2 Bugs", y=1.02);

### Kernal Density of Classes w.r.t one variable

In [ ]:
fig = plt.figure(figsize=(12,8))
sns.kdeplot(data_for_viz_unscaled.loc[(data_for_viz_unscaled['Bugs'] == 0), 'rfc'], color='g', shade=True, Label='0 Bugs') 
sns.kdeplot(data_for_viz_unscaled.loc[(data_for_viz_unscaled['Bugs'] == 1), 'rfc'], color='orange', shade=True, Label='1 Bug') 
sns.kdeplot(data_for_viz_unscaled.loc[(data_for_viz_unscaled['Bugs'] == 2), 'rfc'], color='r', shade=True, Label='> 2 Bugs') 

plt.xlabel('Request for Comments/Change - rfc') 
plt.ylabel('Probability Density')
plt.title("Request for Comments/Change - rfc & Class Probability Density")

plt.xlim(-100, 1000);

In [ ]:
fig = plt.figure(figsize=(12,8))
sns.kdeplot(data_for_viz_unscaled.loc[(data_for_viz_unscaled['Bugs'] == 0), 'cbo'], color='g', shade=True, Label='0 Bugs') 
sns.kdeplot(data_for_viz_unscaled.loc[(data_for_viz_unscaled['Bugs'] == 1), 'cbo'], color='orange', shade=True, Label='1 Bug') 
sns.kdeplot(data_for_viz_unscaled.loc[(data_for_viz_unscaled['Bugs'] == 2), 'cbo'], color='r', shade=True, Label='> 2 Bugs') 

plt.xlabel('Coupling between objects - cbo') 
plt.ylabel('Probability Density')
plt.title("Coupling between objects - cbo & Class Probability Density")

plt.xlim(-20, 200);

In [ ]:
fig = plt.figure(figsize=(12,8))
sns.kdeplot(data_for_viz_unscaled.loc[(data_for_viz_unscaled['Bugs'] == 0), 'fanOut'], color='g', shade=True, Label='0 Bugs') 
sns.kdeplot(data_for_viz_unscaled.loc[(data_for_viz_unscaled['Bugs'] == 1), 'fanOut'], color='orange', shade=True, Label='1 Bug') 
sns.kdeplot(data_for_viz_unscaled.loc[(data_for_viz_unscaled['Bugs'] == 2), 'fanOut'], color='r', shade=True, Label='> 2 Bugs') 

plt.xlabel('Fan-Out - fanOut') 
plt.ylabel('Probability Density')
plt.title("Fan-Out - fanOut & Class Probability Density")

plt.xlim(-20, 75);

In [ ]:
fig = plt.figure(figsize=(12,8))
sns.kdeplot(data_for_viz_unscaled.loc[(data_for_viz_unscaled['Bugs'] == 0), 'wmc'], color='g', shade=True, Label='0 Bugs') 
sns.kdeplot(data_for_viz_unscaled.loc[(data_for_viz_unscaled['Bugs'] == 1), 'wmc'], color='orange', shade=True, Label='1 Bug') 
sns.kdeplot(data_for_viz_unscaled.loc[(data_for_viz_unscaled['Bugs'] == 2), 'wmc'], color='r', shade=True, Label='> 2 Bugs') 

plt.xlabel('Weighted Method Per Class - wmc') 
plt.ylabel('Probability Density')
plt.title("Weighted Method Per Class - wmc & Class Probability Density")

plt.xlim(-100, 600);

In [ ]:
fig = plt.figure(figsize=(12,8))
sns.kdeplot(data_for_viz_unscaled.loc[(data_for_viz_unscaled['Bugs'] == 0), 'numberOfLinesOfCode'], color='g', shade=True, label='0 Bugs') 
sns.kdeplot(data_for_viz_unscaled.loc[(data_for_viz_unscaled['Bugs'] == 1), 'numberOfLinesOfCode'], color='orange', shade=True, label='1 Bug') 
sns.kdeplot(data_for_viz_unscaled.loc[(data_for_viz_unscaled['Bugs'] == 2), 'numberOfLinesOfCode'], color='r', shade=True, label='> 2 Bugs') 

plt.xlabel('numberOfLinesOfCode') 
plt.ylabel('Probability Density')
plt.title("numberOfLinesOfCode & Class Probability Density")
plt.xlim(-500, 2000);

#### Scatter 3D Cross Plot

In [ ]:
fig = plt.figure(figsize=(14,10))
ax = fig.gca(projection='3d')

ax.scatter(data_for_viz_unscaled[data_for_viz_unscaled['Bugs'] == 0]['fanOut'], 
           data_for_viz_unscaled[data_for_viz_unscaled['Bugs'] == 0]['rfc'], 
           data_for_viz_unscaled[data_for_viz_unscaled['Bugs'] == 0]['cbo'], c='g', 
           s=data_for_viz_unscaled[data_for_viz_unscaled["Bugs"] == 0]['cbo']/3, alpha=0.7, label='0 Bugs')

ax.scatter(data_for_viz_unscaled[data_for_viz_unscaled['Bugs'] == 1]['fanOut'], 
           data_for_viz_unscaled[data_for_viz_unscaled['Bugs'] == 1]['rfc'], 
           data_for_viz_unscaled[data_for_viz_unscaled['Bugs'] == 1]['cbo'],
           s=data_for_viz_unscaled[data_for_viz_unscaled["Bugs"] == 1]['cbo']/3, alpha=0.7, c='orange', label='1 Bug')

ax.scatter(data_for_viz_unscaled[data_for_viz_unscaled['Bugs'] == 2]['fanOut'], 
           data_for_viz_unscaled[data_for_viz_unscaled['Bugs'] == 2]['rfc'], 
           data_for_viz_unscaled[data_for_viz_unscaled['Bugs'] == 2]['cbo'],
           s=data_for_viz_unscaled[data_for_viz_unscaled["Bugs"] == 2]['cbo']/3, alpha=0.7, c='r', label='> 2 Bugs')
ax.legend()

ax.set_xlabel('fanOut')
ax.set_ylabel('rfc')
ax.set_zlabel('cbo')
ax.set_title("fanOut VS rfc VS cbo");

In [ ]:
plt.figure(figsize=(14,10))
plt.scatter(data_for_viz_unscaled[data_for_viz_unscaled["Bugs"] == 0]['numberOfLinesOfCode'], 
            data_for_viz_unscaled[data_for_viz_unscaled["Bugs"] == 0]['wmc'], label='0 Bugs', 
            s=data_for_viz_unscaled[data_for_viz_unscaled["Bugs"] == 0]['wmc']/10, c='g',
           alpha=0.5)

plt.scatter(data_for_viz_unscaled[data_for_viz_unscaled["Bugs"] == 1]['numberOfLinesOfCode'], 
            data_for_viz_unscaled[data_for_viz_unscaled["Bugs"] == 1]['wmc'], label='1 Bugs', 
            s=data_for_viz_unscaled[data_for_viz_unscaled["Bugs"] == 1]['wmc']/10, c='orange',
           alpha=0.5)

plt.scatter(data_for_viz_unscaled[data_for_viz_unscaled["Bugs"] == 2]['numberOfLinesOfCode'], 
            data_for_viz_unscaled[data_for_viz_unscaled["Bugs"] == 2]['wmc'], label='> 2 Bugs', 
            s=data_for_viz_unscaled[data_for_viz_unscaled["Bugs"] == 2]['wmc']/10, c='r',
           alpha=0.5)

plt.legend()
plt.xlabel('numberOfLinesOfCode')
plt.ylabel('wmc')
plt.title("numberOfLinesOfCode VS wmc");

In [ ]:
fig = plt.figure(figsize=(10,6))

    
ax = sns.barplot(x="Bugs", y="rfc", data=for_pair_plot, alpha=0.7, palette=['g', 'orange', 'r'])
plt.title("Bugs & numberOfLinesOfCode");

In [ ]:
fig = plt.figure(figsize=(10,6))
ax = sns.barplot(x="Bugs", y="cbo", data=for_pair_plot, alpha=0.7, palette=['g', 'orange', 'r'])
plt.title("Bugs & Coupling Between Objects - cbo");

In [ ]:
fig = plt.figure(figsize=(10,6))
ax = sns.barplot(x="Bugs", y="fanOut", data=for_pair_plot, alpha=0.7, palette=['g', 'orange', 'r'])
plt.title("Bugs & Fan Out");

In [ ]:
fig = plt.figure(figsize=(10,6))
ax = sns.barplot(x="Bugs", y="wmc", data=for_pair_plot, alpha=0.7, palette=['g', 'orange', 'r'])
plt.title("Bugs & Weighted Method Per Class - wmc");

In [ ]:
fig = plt.figure(figsize=(10,6))
ax = sns.barplot(x="Bugs", y="numberOfLinesOfCode", data=for_pair_plot, alpha=0.7, palette=['g', 'orange', 'r'])
plt.title("Bugs & numberOfLinesOfCode");

## Observation:
It is very clear from all the plots above that the larger the number of negative properties (not necessarily negative but intuitively relates to a risk prone or a bug susceptible software property) the larger the probability of having more bugs.

# Clustering Analysis
### UMAP Dimensionality reduction algorithm
This is an alogorithm that was developed in 2018 which competes with state-of-the-art dimensionality reduction algorithms in its fast execution performance and the ability to project the dimensions without losing a lot of information. 

Feel free to explore benchmarking against other algorithms here: 
https://umap-learn.readthedocs.io/en/latest/benchmarking.html

*McInnes, Leland, Healy, John, & James. (2018, December 06). UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction. Retrieved from https://arxiv.org/abs/1802.03426*

In [ ]:
reducer = umap.UMAP(verbose=False)
embedding = reducer.fit_transform(X_train_scaled)
embedding.shape

In [ ]:
embedding = pd.DataFrame(embedding)
embedding.head()

#### Next plot is colored by K-Means predictions

In [ ]:
kmeans_umap = KMeans(n_clusters=3)
kmeans_umap.fit(embedding)
preds_kmeans_umap = kmeans_umap.predict(embedding)

fig, ax = plt.subplots(figsize=(14,8))

plt.scatter(embedding[0], embedding[1], c=preds_kmeans_umap, s=10, cmap='viridis')
cbar = plt.colorbar(boundaries=np.arange(4)-0.5)
cbar.set_ticks(np.arange(3))
cbar.set_ticklabels(np.unique(preds_kmeans_umap))

plt.scatter(kmeans_umap.cluster_centers_[:, 0], kmeans_umap.cluster_centers_[:, 1], c='red', s=150, alpha=0.7);

plt.title('# Bugs & Software Properties & K-Means Centers | 2D - UMAP', fontsize=14);

In [ ]:
print("K-Means Accuracy Score:", accuracy_score(preds_kmeans_umap, y_train))

### Note: 
This accuracy on the training set is not a surprise! Take a look at the plot below.

Note that there is no relationship between the number of bugs and the clusters set by K-Means predictor. 0,1, and 2 in the plot are just the specified number of clusters for K-Means.

To clarify the confusion
#### Next Plot is colored by actual values (y_train) and K-Means centers are included

In [ ]:
fig, ax = plt.subplots(figsize=(14,8))

plt.scatter(embedding[0], embedding[1], c=y_train, s=10, cmap='viridis')
cbar = plt.colorbar(boundaries=np.arange(4)-0.5)
cbar.set_ticks(np.arange(3))
cbar.set_ticklabels(np.unique(y_train))

plt.scatter(kmeans_umap.cluster_centers_[:, 0], kmeans_umap.cluster_centers_[:, 1], c='red', s=150, alpha=0.7);

plt.title('# Bugs & Software Properties & K-Means Centers | 2D - UMAP', fontsize=14);

### PCA Dimensionality reduction algorithm

#### Next plot is colored by actual values (y_train)
#### Note:
The plot was zoomed due to extreme outliers

In [ ]:
pca = PCA(n_components=2)
principalComponents = pca.fit_transform(X_train)

principalComponents = pd.DataFrame(principalComponents)

In [ ]:
fig, ax = plt.subplots(figsize=(14,8))

plt.scatter(principalComponents[0], principalComponents[1], c=y_train, s=10, cmap='viridis')
cbar = plt.colorbar(boundaries=np.arange(4)-0.5)
cbar.set_ticks(np.arange(3))
cbar.set_ticklabels(np.unique(y_train))
ax.set_xlim(-500, 10000);
ax.set_ylim(-500, 6000);

plt.title('# Bugs & Software Properties | 2D - PCA (Zoomed)', fontsize=14);

#### Next Plot is colored by K-Means cluster predictions and not actual values (y_train)

In [ ]:
kmeans_pca = KMeans(n_clusters=3)
kmeans_pca.fit(principalComponents)
preds_kmeans_pca = kmeans_pca.predict(principalComponents)

fig, ax = plt.subplots(figsize=(14,8))

plt.scatter(principalComponents[0], principalComponents[1], c=preds_kmeans_pca, s=7, cmap='viridis')
cbar = plt.colorbar(boundaries=np.arange(4)-0.5)
cbar.set_ticks(np.arange(3))
cbar.set_ticklabels(np.unique(preds_kmeans_pca))

plt.scatter(kmeans_pca.cluster_centers_[:, 0], kmeans_pca.cluster_centers_[:, 1], c='red', s=150, alpha=0.7);

# ax.set_xlim(-500, 10000);
# ax.set_ylim(-500, 6000);

plt.title('# Bugs & Software Properties | 2D - PCA', fontsize=14);

print("K-Means Accuracy Score Using PCA:", accuracy_score(preds_kmeans_pca, y_train))

### Note
It is obvious to spot that K-Means had a high accuracy on the training set but definitely not a generalizable one not just because it's on the training set but mainly, for our analysis, it has almost excluded a whole class (**Recall: Notes on class balances in the dataset**) if we took a look at the actualy distribution of classes when if was plotted by actual values after PCA reduction and now with the predicted values from K-Means

# Modeling 

#### Models:
- For Multi-Class Classification
    - Random Forest Classifier (Ensemble)
    - K-Nearest Neighbor
    - K-Means (Clustering)

- For Binary Classification
    - AdaBoost (Boosting)
    - Support-Vector Machine
    - Bagging Classifier

- Neural Network (FCNN)

All of these models either:
- need scaling for better generalized results (i.e. SVC, KNN, KMeans, FCNN) as they are distance-based where they are affected by outliers and more importantly (in our case) might favor one feature against another.
- not affected by scaling if the weak learner is a tree-based classifier for AdaBoost, Bagging, or RFC

#### Scale Features seperately in each dataset (NOT the cross validation one; cv dataset should be scaled with each fold in hyper-parameter tuning)

In [ ]:
X_train_scaled = pd.DataFrame(StandardScaler().fit_transform(X_train.values), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(StandardScaler().fit_transform(X_test.values), columns=X_test.columns, index=X_test.index)

print("Train:", X_train.shape, y_train.shape,
      "Test:", X_test.shape, y_test.shape,
      "Cross Validation", X_cv.shape, y_cv.shape)

#### Modeling Function

In [ ]:
def Modeler(model, X_train, y_train, X_cv, y_cv, params, scale=True, n_jobs=True, pca=False):
    
    if scale & pca:
        pipeline = Pipeline([
            ('scale', StandardScaler()),
            ('dim_reduct', PCA(n_components=4)),
            ('clf', model())])
        print("Scaling and dim reduct...")
        
    elif scale:
        pipeline = Pipeline([ ('scale', StandardScaler()), ('clf', model())])
        
    elif pca:
        pipeline = Pipeline([
        ('dim_reduct', PCA(n_components=pca)),
        ('clf', model())])
    else:
        pipeline = Pipeline([('clf', model())])
        
    grid = GridSearchCV(pipeline, param_grid = params, cv=3, n_jobs=4, refit=True)
    grid.fit(X_cv, y_cv)
    
    best_prameters = {}
    for k, v in grid.best_params_.items():
        best_prameters[k[5:]] = v

    if n_jobs:
        model = model(**best_prameters, n_jobs=4)
    else:
        model = model(**best_prameters)
   
    model.fit(X_train, y_train)
    
    print("Best Parameters for model:", best_prameters)

    return {
        "model": model, 
        "best_params":best_prameters
    }

#### Evaluator Function

#### Note 
- We will value all classes in the same way without any dominance from one class on the metric. Therefore, we will use Macro-Averaging for both ROC and F1-Score. This will tell us that our scores will be much lower than expected due to the imbalance (**More details on that at the end**) Event though that ROC is no that sensitive to imbalances, F1-Score can help us a lot here
- We can use AUCROC by one-vs-all Technique

In [ ]:
def model_eval(model, X_test, y_test, acc=True, auc_=True, plot_conf=True, plot_auc=True, multi=True, f1=True, prop=False):
    
    res = [None, None, None] # Accuracy, auc_roc, f1_score
    y_pred = model.predict(X_test)
    
    if prop:
        y_pred = np.where(y_pred > 0.5, 1, 0)
    
    if f1:
        f_sc = f1_score(y_test, y_pred, average='macro')
        res[2] = f_sc
        
    if acc:
        res[0] = accuracy_score(y_test, y_pred)
#         print('accuracy:', res[0] * 100 ,'%')

    if auc_:
        lb = LabelBinarizer()
        lb.fit(y_test)

        truth = lb.transform(y_test)
        y_pred_encode = lb.transform(y_pred)

        res[1] = roc_auc_score(truth, y_pred_encode, average = 'macro')
#         print('ROC:', res[1] * 100 ,'%')

    if plot_conf:
        fig, ax = plt.subplots(figsize=(10,6))

        conf = confusion_matrix(y_test, y_pred, labels=np.unique(y_test))
        sns.heatmap(conf, cmap= 'Set1', annot=True, cbar=False)
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.title('Confusion Matrix');
    
    if plot_auc:
        fig, ax = plt.subplots(figsize=(10,6))

        if not auc_:
            lb = LabelBinarizer()
            lb.fit(y_test)

            truth = lb.transform(y_test)
            y_pred_encode = lb.transform(y_pred)

        fpr = [None, None, None]
        tpr = [None, None, None]
        auc_of_roc = [None, None, None]
        
        n_classes = 1 #(0, 1)
        if multi:
            n_classes = 3
        
        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve(truth[:, i], y_pred_encode[:, i])
            auc_of_roc[i] = auc(fpr[i], tpr[i])
            
        plt.plot(fpr[0], tpr[0], label='0 - area under ROC = %0.3f' % auc_of_roc[0])
        if multi:
            plt.plot(fpr[1], tpr[1], label='1 - area under ROC = %0.3f' % auc_of_roc[1])
            plt.plot(fpr[2], tpr[2], label='2 - area under ROC = %0.3f' % auc_of_roc[2])

        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title("ROC AUC");
        plt.legend()

    return res


#### Under Sampler Function

In [ ]:
def under_sample(X_train, y_train, n=573):
    train_ = pd.concat([X_train, y_train], axis=1)
    df_0 = train_[train_.iloc[:, -1] == 0].sample(n= n)
    df_1 = train_[train_.iloc[:, -1] == 1].sample(n= n)

    train_ = pd.concat([df_0, df_1], axis=0).sample(frac=1.0)
    X_train_under_sampled = train_.iloc[:, 0:-1]
    y_train_under_sampled = train_.iloc[:, -1]
    return X_train_under_sampled, y_train_under_sampled

## Base-Line Classifier
We have to give a better accuracy, roc, or f1-score if we want our classifier to be better than a mere guessing

In [ ]:
dmc = DummyClassifier(strategy="most_frequent")
dmc.fit(X_train_scaled, y_train)

model_scoring = model_eval(dmc, X_test_scaled, y_test)

scores = {}
scores["Dummy Classifier | Multi"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}

In [ ]:
dmc = DummyClassifier(strategy="most_frequent")
dmc.fit(X_train_scaled, y_train)

model_scoring = model_eval(dmc, X_test_scaled, y_test, multi=False)

scores["Dummy Classifier | Binary"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}

## Multi-Classifiers

### Random Forest Classifier

In [ ]:
rfc_params = {
    'clf__n_estimators'      : [200, 500, 1000],
    'clf__max_depth'         : [10, 20, 50],
    'clf__max_features': [1.0, 0.7, 0.4],
    'clf__criterion' :['gini', 'entropy']
}

rfc = Modeler(RandomForestClassifier, X_train_scaled, y_train, X_cv, y_cv, rfc_params, scale=True) 

In [ ]:
model_scoring = model_eval(rfc["model"], X_test_scaled, y_test)
scores["Random Forest | Multi"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}

### K-Nearest Neighbor

In [ ]:
knn_params = {
    'clf__n_neighbors': [3, 5, 11, 17], # Try not to put even numbers
    'clf__weights': ['uniform', 'distance'],
    'clf__metric': ['euclidean', 'manhattan'],
}

knn = Modeler(KNeighborsClassifier, X_train_scaled, y_train, X_cv, y_cv, knn_params, scale=True)

In [ ]:
model_scoring = model_eval(knn["model"], X_test_scaled, y_test)
scores["K-Nearest Neighbor | Multi"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}

In [ ]:
kmeans_params = {
    'clf__max_iter': [200, 500, 1000], # Try not to put even numbers
    'clf__n_init': [10, 30]
}

kmeans = Modeler(KMeans, X_train_scaled, y_train, X_cv, y_cv, kmeans_params, scale=True)

In [ ]:
model_scoring = model_eval(kmeans["model"], X_test_scaled, y_test)
scores["K-Means | Multi"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}


### Bagging Classifier (Extra for multi-class)

In [ ]:
bagg_params = {
    'clf__max_samples': [0.3, 0.5, 0.7, 1.0],
    'clf__bootstrap': [True, False],
    'clf__bootstrap_features': [True, False]
}

bagg = Modeler(BaggingClassifier, X_train_scaled, y_train, X_cv, y_cv, bagg_params, scale=True)

In [ ]:
model_scoring = model_eval(bagg["model"], X_test_scaled, y_test)
scores["Bagging Classifier | Multi"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}

### Bagging Classifier (with PCA 4D)

In [ ]:
bagg_params = {
    'clf__max_samples': [0.3, 0.5, 0.7, 1.0],
    'clf__bootstrap': [True, False],
    'clf__bootstrap_features': [True, False]
}

bagg = Modeler(BaggingClassifier, X_train_scaled, y_train, X_cv, y_cv, bagg_params, scale=True, pca=4)


In [ ]:
model_scoring = model_eval(bagg["model"], X_test_scaled, y_test)

scores["Bagging Classifier | Multi | PCA"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}
model_scoring

## Binary-Classifiers

In [ ]:
# We do not need confusing variables such as the class name; We can use it as an index or leave it.
X_binary = df.iloc[:, 1:-2]
y_binary = df["bugs"]

# Warning: Don't Run Twice
y_binary = y_binary.replace(y_binary.where(y_binary > 1), 1)

print("X:", X.shape)
print("y:", y.shape)

In [ ]:
unique, counts = np.unique(y_binary, return_counts=True)
print("Classes:", unique.tolist())
print("Counts:", counts.tolist())

plt.bar(unique, counts)
plt.title("#Bugs VS Occurrences")
plt.xticks(range(len(unique)))
plt.ylabel("Occurrences")
plt.xlabel("# Bugs");

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_binary, y_binary, test_size=0.3)
X_cv, X_test, y_cv, y_test = train_test_split(X_test, y_test, test_size=0.5)
print("Train:", X_train.shape, y_train.shape,
      "Test:", X_test.shape, y_test.shape,
      "Cross Validation", X_cv.shape, y_cv.shape)

X_train_scaled = pd.DataFrame(StandardScaler().fit_transform(X_train.values), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(StandardScaler().fit_transform(X_test.values), columns=X_test.columns, index=X_test.index)

#### Warning: X_train, X_test,..., y_cv. has been overridden

### AdaBoost

In [ ]:
# AdaBoostClassifier
ada_params = {
    'clf__learning_rate': [0.1, 0.5, 1.0],
    'clf__n_estimators': [100, 200]
}

ada = Modeler(AdaBoostClassifier, X_train_scaled, y_train, X_cv, y_cv, ada_params, scale=True, n_jobs=False)

In [ ]:
model_scoring = model_eval(ada["model"], X_test_scaled, y_test, multi=False)
scores["AdaBoost Classifier | Binary"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}

### Support-Vector Machine

In [ ]:
# SVC()
svc_params = {
    'clf__C': [0.1, 1, 10],
    'clf__kernel': ['linear', 'rbf'],
    'clf__gamma':[1, 0.01, 0.001]
}

svc = Modeler(SVC, X_train_scaled, y_train, X_cv, y_cv, svc_params, scale=True, n_jobs=False)

In [ ]:
model_scoring = model_eval(svc["model"], X_test_scaled, y_test, multi=False)
scores["Support-Vector Machine | Binary"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}

### Bagging Classifier

In [ ]:
bagg_params = {
    'clf__max_samples': [0.3, 0.5, 0.7, 1.0],
    'clf__bootstrap': [True, False],
    'clf__bootstrap_features': [True, False]
}

bagg = Modeler(BaggingClassifier, X_train_scaled, y_train, X_cv, y_cv, bagg_params, scale=True)

In [ ]:
model_scoring = model_eval(bagg["model"], X_test_scaled, y_test, multi=False)
scores["Bagging Classifier | Binary"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}

## Neural Network

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from keras import optimizers


In [ ]:
def nn_modeler(input_dim, layer_1_dim, layer_2_dim, lr, out):
    model = Sequential()
    model.add(Dense(layer_1_dim, input_dim=input_dim, activation='relu'))
    model.add(Dense(layer_2_dim, activation='relu'))
    model.add(Dense(out, activation='sigmoid'))
    SGD = optimizers.SGD(lr=lr)

    model.compile(loss='binary_crossentropy', optimizer = SGD)
    return model

In [ ]:

NN_scores = {}
n_split=3
epochs = [30, 50] # Take a lot of time
batch_sizes = [32, 64] # Take a lot of time
first_layers = [30, 50]
second_layers = [10, 20]
lrs = [0.001, 0.1]
iters = 0
for epoch in epochs:
    for batch_size in batch_sizes:
        for first_layer in first_layers:
            for second_layer in second_layers:
                for lr in lrs:
                    # K-FOLDS
                    fcnn = None
                    accs = []
                    aurocs = []
                    f1s = []
                    for train_indicies, test_indicies in StratifiedKFold(n_split).split(X_cv, y_cv):
                        X_train_fold, y_train_fold, X_test_fold, y_test_fold = X_cv.iloc[train_indicies], y_cv.iloc[train_indicies], X_cv.iloc[test_indicies], y_cv.iloc[test_indicies]
                        X_train_fold = pd.DataFrame(StandardScaler().fit_transform(X_train_fold.values), columns=X_train_fold.columns, index=X_train_fold.index)
                        X_test_fold = pd.DataFrame(StandardScaler().fit_transform(X_test_fold.values), columns=X_test_fold.columns, index=X_test_fold.index)

                        fcnn = nn_modeler(17, first_layer, second_layer, lr, 1)
                        fcnn.fit(X_train_fold, y_train_fold, epochs=epoch, batch_size=batch_size, verbose=False)
                        model_scoring = model_eval(fcnn, X_test_scaled, y_test, plot_auc=False, plot_conf=False, prop=True)
                        accs.append(model_scoring[0])
                        aurocs.append(model_scoring[1])
                        f1s.append(model_scoring[2])
                    
                    info = "ep:" + str(epoch) + " bs:" + str(batch_size) + " f_lyr:" + str(first_layer) + " s_lyr:" + str(second_layer) + " lr:" + str(lr)

                    NN_scores[info] = {"Accuracy": np.mean(accs), "ROC": np.mean(aurocs), "F1": np.mean(f1s)}
                    iters +=1
                    print(str(iters) +'..', end='')

In [ ]:
NN_scores = pd.DataFrame.from_dict(NN_scores).T
NN_scores = NN_scores.sort_values('ROC', ascending=False)

In [ ]:
NN_scores.iloc[:10,:]

##### Best Params

- Epochs: 30
- Batch Size: 32
- First Hidden Layer neurons: 50
- Second Hidden Layer neurons: 10
- Learning Rate: 0.1

In [ ]:
fcnn = nn_modeler(17, 50, 10, 0.1, 1)
fcnn.fit(X_train_scaled, y_train, epochs=30, batch_size=32, verbose=False)

model_scoring = model_eval(fcnn, X_test_scaled, y_test, plot_auc=False, plot_conf=False, prop=True)


scores["Neural Network | Binary"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}

#### Note:
All models achieved a similar or less auc rocs/f1 without scaling (tried it for a sanity check)

In [ ]:
score_df = pd.DataFrame.from_dict(scores).T
score_df= score_df.sort_values('ROC', ascending=False)

In [ ]:
score_df

# Data-Driven Model Optimization

##### Pick the 3 highest models

## Under Sampling

Redcuing the number of samples in the majority class randomly so that the two classes are balanced and then testing it on an imbalanced real-world-like dataset (test set)

#### Neural Network

This modle was also trained on a PCAed training set with/out under-sampling

In [ ]:
X_train_under_sampled, y_train_under_sampled = under_sample(X_train_scaled, y_train, n=(y_train==1).sum())

# print((y_train_under_sampled == 1).sum())
# print((y_train_under_sampled == 0).sum())

fcnn = nn_modeler(17, 50, 10, 0.1, 1)
fcnn.fit(X_train_under_sampled, y_train_under_sampled, epochs=30, batch_size=32, verbose=False)

In [ ]:
model_scoring = model_eval(fcnn, X_test_scaled, y_test, plot_auc=False, plot_conf=False, prop=True)
scores["Neural Network | Binary | USam"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}
model_scoring

#### AdaBoost

In [ ]:
# X_cv_scaled = pd.DataFrame(StandardScaler().fit_transform(X_cv.values), columns=X_cv.columns, index=X_cv.index)
X_cv_under_sampled, y_cv_under_sampled = under_sample(X_cv, y_cv, n=(y_cv == 1).sum())

ada_params = {
    'clf__learning_rate': [0.1, 0.5, 1.0],
    'clf__n_estimators': [100, 200]
}

ada = Modeler(AdaBoostClassifier, X_train_under_sampled, y_train_under_sampled, X_cv_under_sampled, y_cv_under_sampled,
              ada_params, scale=True, n_jobs=False)

In [ ]:
model_scoring = model_eval(ada["model"], X_test_scaled, y_test, multi=False, plot_auc=False, plot_conf=False)
scores["AdaBoost Classifier | Binary | Usam"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}
model_scoring

#### Bagging Classifier

In [ ]:
bagg_params = {
    'clf__max_samples': [0.3, 0.5, 0.7, 1.0],
    'clf__bootstrap': [True, False],
    'clf__bootstrap_features': [True, False]
}

bagg = Modeler(BaggingClassifier, X_train_under_sampled, y_train_under_sampled, X_cv_under_sampled, y_cv_under_sampled, 
               bagg_params, scale=True)

In [ ]:
model_scoring = model_eval(bagg["model"], X_test_scaled, y_test, multi=False, plot_auc=False, plot_conf=False)
scores["Bagging Classifier | Binary | Usam"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}
model_scoring

In [ ]:
score_df = pd.DataFrame.from_dict(scores).T
score_df= score_df.sort_values('ROC', ascending=False)

In [ ]:
score_df

## Over Sampling

In [ ]:
from imblearn.over_sampling import SMOTE
X_train_over_sampled, y_train_over_sampled = SMOTE().fit_sample(X_train_scaled, y_train)
X_cv_over_sampled, y_cv_over_sampled = SMOTE().fit_sample(X_cv, y_cv)

In [ ]:
# print((y_train_over_sampled == 1).sum())
# print((y_train_over_sampled == 0).sum())

# print((y_cv_over_sampled == 1).sum())
# print((y_cv_over_sampled == 0).sum())

X_train_over_sampled = pd.DataFrame(X_train_over_sampled)
X_cv_over_sampled = pd.DataFrame(X_cv_over_sampled)

y_train_over_sampled = pd.Series(y_train_over_sampled)
y_cv_over_sampled = pd.Series(y_cv_over_sampled)

#### Neural Network

In [ ]:
fcnn = nn_modeler(17, 50, 10, 0.1, 1)
fcnn.fit(X_train_scaled, y_train, epochs=30, batch_size=32, verbose=False)

In [ ]:
model_scoring = model_eval(fcnn, X_test_scaled, y_test, plot_auc=False, plot_conf=False, prop=True)
scores["Neural Network | Binary | OSam"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}
model_scoring

#### AdaBoost

In [ ]:
ada_params = {
    'clf__learning_rate': [0.1, 0.5, 1.0],
    'clf__n_estimators': [100, 200]
}

ada = Modeler(AdaBoostClassifier, X_train_over_sampled, y_train_over_sampled, X_cv_over_sampled, y_cv_over_sampled,
              ada_params, scale=True, n_jobs=False)

In [ ]:
model_scoring = model_eval(ada["model"], X_test_scaled, y_test, multi=False, plot_auc=False, plot_conf=False)
scores["AdaBoost Classifier | Binary | Osam"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}
model_scoring

#### Bagging Classifier

In [ ]:
bagg_params = {
    'clf__max_samples': [0.3, 0.5, 0.7, 1.0],
    'clf__bootstrap': [True, False],
    'clf__bootstrap_features': [True, False]
}

bagg = Modeler(BaggingClassifier, X_train_over_sampled, y_train_over_sampled, X_cv_over_sampled, y_cv_over_sampled, 
               bagg_params, scale=True)

In [ ]:
model_scoring = model_eval(bagg["model"], X_test_scaled, y_test, multi=False, plot_auc=False, plot_conf=False)
scores["Bagging Classifier | Binary | Osam"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}
model_scoring

In [ ]:
score_df = pd.DataFrame.from_dict(scores).T
score_df= score_df.sort_values('ROC', ascending=False)

In [ ]:
score_df

## With Feature Selection

#### Refer back to feature selection above, features selected were:
    - rfc
    - cbo
    - fanOut
    - wmc
    - numberOfLinesOfCode

In [ ]:
X_train_scaled_featured = X_train_scaled[['rfc', 'cbo', 'fanOut', 'wmc', 'numberOfLinesOfCode']]
X_test_scaled_featured = X_test_scaled[['rfc', 'cbo', 'fanOut', 'wmc', 'numberOfLinesOfCode']]
X_cv_featured = X_cv[['rfc', 'cbo', 'fanOut', 'wmc', 'numberOfLinesOfCode']]

X_train_scaled_featured_underS = X_train_under_sampled[['rfc', 'cbo', 'fanOut', 'wmc', 'numberOfLinesOfCode']]
X_cv_featured_underS = X_cv_under_sampled[['rfc', 'cbo', 'fanOut', 'wmc', 'numberOfLinesOfCode']]

#### Neural Network

In [ ]:
fcnn = nn_modeler(5, 50, 10, 0.1, 1)
fcnn.fit(X_train_scaled_featured, y_train, epochs=30, batch_size=32, verbose=False)

In [ ]:
model_scoring = model_eval(fcnn, X_test_scaled_featured, y_test, plot_auc=False, plot_conf=False, prop=True)
scores["Neural Network | Binary | Featured"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}
model_scoring

In [ ]:
ada_params = {
    'clf__learning_rate': [0.1, 0.5, 1.0],
    'clf__n_estimators': [100, 200]
}

ada = Modeler(AdaBoostClassifier, X_train_scaled_featured, y_train, X_cv_featured, y_cv,
              ada_params, scale=True, n_jobs=False)

In [ ]:
model_scoring = model_eval(ada["model"], X_test_scaled_featured, y_test, multi=False, plot_auc=False, plot_conf=False)
scores["AdaBoost Classifier | Binary | Featured"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}
model_scoring

In [ ]:
bagg_params = {
    'clf__max_samples': [0.3, 0.5, 0.7, 1.0],
    'clf__bootstrap': [True, False],
    'clf__bootstrap_features': [True, False]
}

bagg = Modeler(BaggingClassifier, X_train_scaled_featured, y_train, X_cv_featured, y_cv, 
               bagg_params, scale=True)

In [ ]:
model_scoring = model_eval(bagg["model"], X_test_scaled_featured, y_test, multi=False, plot_auc=False, plot_conf=False)
scores["Bagging Classifier | Binary | Featured"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}
model_scoring

#### Feature Selection with Under Sampling

In [ ]:
# X_train_scaled_featured_underS
# X_cv_featured_underS

fcnn = nn_modeler(5, 50, 10, 0.1, 1)
fcnn.fit(X_train_scaled_featured_underS, y_train_under_sampled, epochs=30, batch_size=32, verbose=False)

In [ ]:
model_scoring = model_eval(fcnn, X_test_scaled_featured, y_test, plot_auc=False, plot_conf=False, prop=True)
scores["Neural Network | Binary | Featured | USam"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}
model_scoring

In [ ]:
ada_params = {
    'clf__learning_rate': [0.1, 0.5, 1.0],
    'clf__n_estimators': [100, 200]
}

ada = Modeler(AdaBoostClassifier, X_train_scaled_featured_underS, y_train_under_sampled, X_cv_featured_underS, y_cv_under_sampled,
              ada_params, scale=True, n_jobs=False)

In [ ]:
model_scoring = model_eval(ada["model"], X_test_scaled_featured, y_test, multi=False, plot_auc=False, plot_conf=False)
scores["AdaBoost Classifier | Binary | Featured | USam"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}
model_scoring

In [ ]:
bagg_params = {
    'clf__max_samples': [0.3, 0.5, 0.7, 1.0],
    'clf__bootstrap': [True, False],
    'clf__bootstrap_features': [True, False]
}

bagg = Modeler(BaggingClassifier, X_train_scaled_featured_underS, y_train_under_sampled, X_cv_featured_underS, y_cv_under_sampled, 
               bagg_params, scale=True)

In [ ]:
model_scoring = model_eval(bagg["model"], X_test_scaled_featured, y_test, multi=False, plot_auc=False, plot_conf=False)
scores["Bagging Classifier | Binary | Featured | USam"] = {"Accuracy": model_scoring[0],
                             "ROC": model_scoring[1],
                             "F1-Score": model_scoring[2]}
model_scoring

# Model Quality Metric Table

In [ ]:
score_df = pd.DataFrame.from_dict(scores).T

In [ ]:
multi_clf_scores = score_df[(score_df.index.str.contains('Multi'))]
binary_clf_scores = score_df[(score_df.index.str.contains('Binary') == True)]

#### Base-Line

In [ ]:
score_df[(score_df.index.str.contains('Dummy'))]

## Multi-Classifiers

#### By accuracy

In [ ]:
multi_clf_scores.sort_values('Accuracy', ascending=False)

#### By ROC

In [ ]:
multi_clf_scores.sort_values('ROC', ascending=False)

#### By F1-Score

In [ ]:
multi_clf_scores.sort_values('F1-Score', ascending=False)

#### Best in Accuracy

In [ ]:
multi_clf_scores.sort_values('Accuracy', ascending=False).iloc[:1,:]

#### Best in ROC

In [ ]:
multi_clf_scores.sort_values('ROC', ascending=False).iloc[:1,:]

#### Best in F1-Score

In [ ]:
multi_clf_scores.sort_values('F1-Score', ascending=False).iloc[:1,:]

## Binary Classifiers

#### By Accuracy

In [ ]:
binary_clf_scores.sort_values('Accuracy', ascending=False)

#### By ROC

In [ ]:
binary_clf_scores.sort_values('ROC', ascending=False)

#### By F1-Score

In [ ]:
binary_clf_scores.sort_values('F1-Score', ascending=False)

#### Best in Accuracy

In [ ]:
binary_clf_scores.sort_values('Accuracy', ascending=False).iloc[:1,:]

#### Best in ROC

In [ ]:
binary_clf_scores.sort_values('ROC', ascending=False).iloc[:1,:]

#### Best in F1-Score

In [ ]:
binary_clf_scores.sort_values('F1-Score', ascending=False).iloc[:1,:]

### Notes

##### F1-Score uses macro-averaging:
This is a way to ensure that we're giving a fair weight (i.e. decrease the total metric score when the lower-in-size class is misclassified). This choice was made to ensure that classifying 0 bugs when there are actually 1 or more bugs is extremely unwanted in the real world (e.g. predicting NO cancerous cells - majority - when there're cancerous cells is considered a false-negative which is dangerous). Here, the negative class is having a bug > 0. Releasing the software with a false knowledge that there are no bugs creats an issue, in a sense, Misclassifying a negative sample would be detrimental to business.
      
      
      
##### PCA:
Note that most of the models were trained with/out pca but not added due to the large number of unending combinations (7 classifiers x hyper-parameter tuning combinations x 2 (for pca) x 2 (for umap)).
      
      
  
##### Under Sampling
Under sampling is a technique in which we randomly draw without replacement **n** sample from the majority class where **n = full samples in the minority class** to solve the imbalance in the training dataset where the classifier can focus on both classes without falling into a biased measurements towards one class over another. Then we test it on an unchaged real-world imbalanced dataset and check whether the model has achieved an accepted classifications for each class. we can see the improvement after applying this technique. 



##### Over Sampling
The opposite of under sampling in which we focus our attention on the less abundant class and synthesize some extra points for it. This is done when we talk about SMOT (Synthetic Minority Over-sampling Technique) by taking a sample from the minority class and get the k-nearest neighbors of that sample, draw a vector between the sample and the k-nearest point, multiply it by a random number and add this point to the dataset. 
        
        
        
##### AdaBoost
This is one of the classifiers that also helps with this imbalanced dataset as it uses a technique in which it adds more weight to the samples that were incorrectly classified after the initialization of equal weights for all samples (classifier's weight alpha that is based on the error rate of a chosen sample). Now with the class that was incorrectly classified in this imbalanced dataset, the model will focus on these ones more. 

# Extra

Now let's try to intentionally increase the roc and accuracy and leave f1-score out of the equation, I really want to emphasize more on the importance of f1-score when we're dealing with an imbalanced data and we care about recall

\begin{equation*}
{F_β} = (1 + β^2) + \frac{precision * recall}{(β^2 * precision + recall)}
\end{equation*}

This is the harmonic mean which gives a measure of how good the classifier is w.r.t precision and recall

 ${prediction} = {+}$   ${AND}$   ${actual} = {-}$ 


is more dangerous than:


${prediction} = {-}$   ${AND}$   ${actual} = {+}$ 

where positive = no bugs, and negative = bugs

#### This is a quick illustration of increasing the roc and accuracy
We are going to use an anamoly detection algorithm where it focuses only on the distribution of the dataset and predicts whether this is an outlier or inliner w.r.t a certain threshold 

In [ ]:
from sklearn.ensemble import IsolationForest

isol_forest = IsolationForest(max_samples=X_train.shape[0], contamination=0.005)
isol_forest.fit(X_train, y_train)

preds = isol_forest.predict(X_test)
preds = np.asarray([1 if x == -1 else 0 for x in preds])

print("f1-score:", f1_score(preds, y_test) ,"%")
print("AUC-ROC: ", roc_auc_score(preds, y_test) ,"%")
print("Accuracy:", accuracy_score(preds, y_test) ,"%")
print("pred inliner:", sum(preds == 0))
print("pred outlier:", sum(preds == 1))
print("test inliner:", sum(y_test == 0))
print("test outlier:", sum(y_test == 1))

In [ ]:
counter_1 = 0
for pred, act in zip(preds, y_test):
    if pred == act and pred == 1:
        counter_1+=1
print("number of 1's predicted accuratly: ", counter_1)


Now you can see how the roc fails in giving a good measure of how the model will behave in real life. This is what happens when we focus more (using f1-score compared to roc) on false negatives (predicting positive=0=no_bugs but actual is negative=1=bugs)